In [1]:
import conllu # our id data not just a plain text its complex table format where every word has 10+ columns of language info. This lib saves us from writing a confusing Regex parser to read it
import sklearn_crfsuite
from sklearn_crfsuite import metrics


In [2]:
def load_conllu_data(file_path):
    data_list = [] # this is empty list but in the end it will hold our cleaned data

    with open(file_path, 'r', encoding='utf-8') as f: # openning a file in a readable mod with the utf-8 encoding 

        for sentence in conllu.parse_incr(f): # this is read file sentence by sentence instead of loading all at once into ram
            words = []
            tags = []
            for token in sentence: # we will loop through every token
                words.append(token['form']) # actual word seen in text
                tags.append(token['upos']) # universal part of speech tag    
            data_list.append((words, tags)) # this row func easier to show ["makan", "nasi"], ["verb", "noun"]
    return data_list

In [3]:
train_data = load_conllu_data('id_gsd-ud-train.conllu')
dev_data = load_conllu_data('id_gsd-ud-dev.conllu')
test_data = load_conllu_data('id_gsd-ud-test.conllu')

print(f"Training sentences: {len(train_data)}")
print(f"Dev sentences: {len(dev_data)}")
print(f"Test sentences: {len(test_data)}")
print(f"Sample sentence: {train_data[0]}")

Training sentences: 4477
Dev sentences: 559
Test sentences: 557
Sample sentence: (['Sembungan', 'adalah', 'sebuah', 'desa', 'yang', 'terletak', 'di', 'kecamatan', 'Kejajar', ',', 'kabupaten', 'Wonosobo', ',', 'Jawa', 'Tengah', ',', 'Indonesia', '.'], ['PROPN', 'AUX', 'DET', 'NOUN', 'PRON', 'VERB', 'ADP', 'NOUN', 'PROPN', 'PUNCT', 'NOUN', 'PROPN', 'PUNCT', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT'])


In [ ]:
def word_features(sent, i):
    word = sent[i][0] # get the word to given index
    
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],     # Suffix (Important for Indonesian: -kan, -nya)
        'word[-2:]': word[-2:],     # Suffix (e.g. -an)
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(), # its make some normalization like lowercasing and some controls 
    }
    
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(), # what was the previous word
            '-1:word.istitle()': word1.istitle(), # was previous word a name 
        })
    else:
        features['BOS'] = True # start of sentence we will help tolern sentence usually start with pron or propn

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(), # this will look what is the next word
            '+1:word.istitle()': word1.istitle(),
        })
    else:
        features['EOS'] = True # end of the sentence help learn sentence usually end with PUNCT

    return features

def sent2features(sent):
    return [word_features(sent, i) for i in range(len(sent))] # loop every sentence and run the func we just implement

def sent2labels(sent):
    return [label for token, label in sent] # we will extract tags for train

print("Extracting features for CRF")
X_train = [sent2features(s) for s in train_data]
y_train = [sent2labels(s) for s in train_data]

X_test = [sent2features(s) for s in test_data]
y_test = [sent2labels(s) for s in test_data]

Extracting features for CRF...


ValueError: too many values to unpack (expected 2)

In [ ]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs', # its optimization program for CRF
    c1=0.1, # its a regularization, helps ignore useless feature
    c2=0.1, # again a regularization but this time its prevent overfitting, its memorize training data  
    max_iterations=100, # it will stop after 100 iter 
    all_possible_transitions=True # allow to predic tags it never see in trainning
)

print("Training CRF model")
crf.fit(X_train, y_train) # there is a fit keyword as we not its for trainning

y_pred = crf.predict(X_test)
print(f"CRF Accuracy: {metrics.flat_accuracy_score(y_test, y_pred):.2%}")